# Workshop: Lakeflow Declarative Pipelines

Build a Lakeflow pipeline from scratch: write SQL declarations, add quality expectations, create and run the pipeline in Databricks UI.

| Duration | Format | Difficulty |
|---|---|---|
| 30 min | Hands-on Workshop | Intermediate |

**Prerequisites:** 03 — Lakeflow Pipelines Demo

## Scenario

> *"Build a complete Medallion Pipeline using Lakeflow Declarative Pipelines. First you'll create a pipeline via the Databricks UI (Workshop), then practice writing SQL declarations for Bronze, Silver, and Gold layers (Practice)."*

---

## Learning Objectives

After completing this lab you will be able to:

- Create and configure a Lakeflow Declarative Pipeline in the Databricks UI
- Upload SQL source files and set pipeline variables
- Write `STREAMING TABLE` declarations for Bronze ingestion
- Add data quality expectations (`ON VIOLATION DROP ROW`) to Silver
- Create `MATERIALIZED VIEW` declarations for Gold aggregations
- Verify pipeline results and query the Event Log

---


## Prerequisites

- Cluster running and attached to notebook
- SQL source files available in `materials/lakeflow/`

---

## Section 1: Workshop — Building the Pipeline in UI

Follow the trainer's instructions step by step.

### Step 1: Upload SQL Files
- Upload the SQL transformation files from `materials/lakeflow/` via Databricks UI or Git Folders

### Step 2: Create Pipeline
1. Go to **Workflows → Pipelines → Create Pipeline**
2. Set **Pipeline name**: `lakeflow_pipeline_<your_name>`
3. Set **Catalog**: `retailhub_<your_name>`
4. Set **Target schema**: `<your_name>_lakeflow`
5. Add the uploaded SQL files as source code

### Step 3: Configure Variables
- Add pipeline configuration keys:
 - `customer_path` → path to customer data
 - `order_path` → path to order data
 - `product_path` → path to product data

### Step 4: Run the Pipeline
1. Click **Start** to run the pipeline
2. After first run completes, add a new file to `orders/stream/`
3. Run the pipeline again — observe incremental processing
4. Check the **Event Log** tab for processing metrics

### Step 5: Verify Results
- Query `fact_sales` with joins to `dim_customer`, `dim_product`, `dim_date`
- Check SCD Type 2 history in `silver_customers`

---

## Section 2: Practice — Lakeflow SQL Declarations

Complete the `# TODO` cells in the sections below.

| Task | What to do | Key concept |
|------|-----------|-------------|
| **Task 1** | Write Bronze Declaration | `CREATE OR REFRESH STREAMING TABLE` + `read_files()` |
| **Task 2** | Write Silver with Expectations | `ON VIOLATION DROP ROW` for constraints |
| **Task 3** | Write Gold Declaration | `CREATE OR REFRESH MATERIALIZED VIEW` |
| **Task 4** | Compare ST vs MV | Fill comparison table (Streaming Table vs Materialized View) |
| **Task 5** | Verify Pipeline Results | Query bronze, silver, gold tables |
| **Task 6** | Check Pipeline Event Log | `event_log(TABLE(...))` for data quality metrics |

---

## Detailed Hints

### Task 1: Bronze — STREAMING TABLE
- `CREATE OR REFRESH STREAMING TABLE bronze_orders`
- Source: `STREAM read_files('{path}', format => 'json')`

### Task 2: Silver — Expectations
- Constraints use `EXPECT (condition) ON VIOLATION DROP ROW`
- First constraint: `order_id IS NOT NULL`
- Second constraint: `total_price > 0`
- Source: `STREAM(bronze_orders)`

### Task 3: Gold — MATERIALIZED VIEW
- `CREATE OR REFRESH MATERIALIZED VIEW gold_daily_revenue`
- Materialized Views are recalculated from scratch each run

### Task 4: ST vs MV
| Feature | Streaming Table | Materialized View |
|---------|----------------|-------------------|
| Processing mode | Incremental (append) | Full recompute |
| Best for | Raw/Bronze ingestion | Aggregations/Gold |

### Task 6: Event Log
- `SELECT * FROM event_log(TABLE(catalog.schema.table))`
- Filter by `event_type = 'flow_progress'` for data quality metrics

---

## Summary

In this lab you:
- Created a complete Lakeflow Pipeline via Databricks UI
- Wrote SQL declarations for Bronze (STREAMING TABLE), Silver (with expectations), and Gold (MATERIALIZED VIEW)
- Verified incremental processing with streaming sources
- Queried the Event Log for pipeline monitoring

> **Pro Tip:** `STREAMING TABLE` processes data incrementally (append-only). `MATERIALIZED VIEW` recalculates fully each run. Use `ON VIOLATION DROP ROW` to silently filter invalid rows. `FAIL` stops the pipeline. `EXPECT` without action only logs warnings.

> **What's next:** The Orchestration demo (04) covers multi-task Jobs with dependencies and triggers.

---
## Lab Tasks

---

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Lakeflow pipeline target schema (matches Step 2 config)
user_name = CATALOG.replace(f"{CATALOG_PREFIX}_", "")
user_schema = f"{user_name}_lakeflow"
print(f"Pipeline target schema: {user_schema}")

---

## Section 1: Workshop — Building the Pipeline

In this hands-on workshop, we create a complete Lakeflow pipeline from SQL files — uploading source code, configuring the pipeline in the Databricks UI, running it, and validating the results.

---

### SQL Files Overview

The pipeline source code is organized by medallion layer — each SQL file declares one table or view.

![57bea06e969c45dab4de8bea9ec980](../../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp)

### Step 1: Upload SQL Files

**Option A: Via UI** — Workspace → Users → create `lakeflow_pipeline` folder → upload SQL files

**Option B: Via Git Folders** — Git Folders → Add Git Folder → clone training repository

### Step 2: Create Pipeline

1. **Workflows** → **Jobs & Pipelines** → **Create ETL Pipeline**
2. **Catalog:** `retailhub_<your_name>`
3. **Pipeline Name:** `lakeflow_pipeline_<your_name>`
4. **Target Schema:** `<your_name>_lakeflow`
5. **Source Code:** Add existing assets → choose pipeline root folder and source code folder

<img src="../../../assets/images/fab4fef8e72d4d5786ba818e6c2f73c5.png" width="800">

<img src="../../../assets/images/a4e21cb35d3b45f2ba184877139cdc48.png" width="800">

<img src="../../../assets/images/4e75e96544f142508c3bd8b0b4dbf446.png" width="800">

![dc9c3eea154e4cb29aee62e6edd5bc](../../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp)

### Step 3: Configure Variables

**Configuration** → **Add configuration**:

| Key | Value |
|-----|-------|
| `customer_path` | `/Volumes/<your catalog>/default/datasets/customers` |
| `order_path` | `/Volumes/<your catalog>/default/datasets/orders` |
| `product_path` | `/Volumes/<your catalog>/default/datasets/products/products.parquet/` |

Open settings and go to Pipeline Configuration 

<img src="../../../assets/images/b2fa841a52004939ac2679f9edef8dc3.png" width="800">

Add configuration : 

<img src="../../../assets/images/dc31d5dec7444c1d959b8e1f220c10ce.png" width="800">

<img src="../../../assets/images/6f16ad4f7fd941ebbb4fb286dbb8fbfd.png" width="800">

You should also see a DAG diagram built based on Spark Declarative Pipelines definition

![05f00c9fedb54b0b81ec65bf182a92](../../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp)

### Step 4: Run the Pipeline

Start the pipeline and test incremental processing by adding new data files.

![8d06de8a2a674cc1bc119b5d91b2d1](../../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp)

1. Add new file to folder orders/stream/ from /Volumes/.../default/datasets/demo/ingestion/orders/stream/
2. Run pipeline again
3. Check Event Log - should process only new files

### Step 5: Verify Results

In [ ]:
# Check fact_sales with joins to dimensions
display(spark.sql(f"""
    SELECT 
        f.order_id,
        c.first_name || ' ' || c.last_name AS customer_name,
        p.product_name,
        d.date,
        f.quantity,
        f.net_amount
    FROM {CATALOG}.{user_schema}.fact_sales f
    LEFT JOIN {CATALOG}.{user_schema}.dim_customer c ON f.customer_key = c.customer_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_product p ON f.product_key = p.product_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_date d ON f.order_date_key = d.date_key
    LIMIT 10
"""))

In [ ]:
# Find customers with change history
display(spark.sql(f"""
    SELECT 
        customer_id, first_name, city,
        __START_AT, __END_AT,
        CASE WHEN __END_AT IS NULL THEN 'Current' ELSE 'Historical' END AS status
    FROM {CATALOG}.{user_schema}.silver_customers
    WHERE customer_id IN (
        SELECT customer_id 
        FROM {CATALOG}.{user_schema}.silver_customers 
        GROUP BY customer_id HAVING COUNT(*) > 1
    )
    ORDER BY customer_id, __START_AT
"""))

### Monitoring and Troubleshooting

Common issues encountered when running Lakeflow pipelines and how to resolve them.

| Issue | Cause | Solution |
|---------|-----------|-------------|
| Pipeline hangs | Cluster too small | Increase min workers |
| Missing data | Constraint DROP ROW | Check Data Quality tab |
| Schema mismatch | Schema change | Full refresh |

---

## Section 2: Practice — Lakeflow SQL Declarations

Write and verify Lakeflow SQL syntax for each medallion layer.

---
## Task 1: Write Bronze Declaration ~4 min

Complete the SQL below to create a Bronze streaming table from JSON files.

**What you need to do:** Fill in the blanks:
1. `CREATE OR REFRESH ________ TABLE` → `STREAMING`
2. `STREAM ________(...)` → `read_files`

**Guidance — Task 01**

The goal is to declare a **Bronze streaming table** that ingests raw JSON files using Lakeflow's `read_files` function.

**STREAMING TABLE vs regular table**
A `STREAMING TABLE` processes data incrementally — it tracks what has been read and only picks up new files on subsequent pipeline runs. This is the standard pattern for Bronze ingestion. The `STREAM` keyword before `read_files()` tells Lakeflow to treat the source as a streaming input.

**The `read_files()` function**
`read_files()` is Lakeflow's built-in function for reading files from Volumes or cloud storage. It accepts a path and optional parameters like `format => 'json'`. Inside a pipeline, it replaces the need for `spark.readStream.format("cloudFiles")`.

**Things to think about**
- Why do we use `STREAM read_files(...)` instead of just `read_files(...)`?
- How does Lakeflow know which files are "new" on subsequent runs?

In [ ]:
# Practice: what the Bronze SQL declaration looks like
# (This won't execute outside of a Lakeflow pipeline, but verify the syntax)

bronze_sql = f"""
-- TODO: Complete the Bronze declaration
CREATE OR REFRESH ________ TABLE bronze_orders
AS SELECT * 
FROM STREAM ________('{DATASET_PATH}/orders/stream', format => 'json');
"""

print("Bronze SQL declaration:")
print(bronze_sql)

In [ ]:
# -- Validation --
assert "STREAMING" in bronze_sql.upper(), "Should use STREAMING TABLE"
assert "read_files" in bronze_sql.lower(), "Should use read_files()"
print("Task 1 OK: Bronze declaration syntax correct")

---
## Task 2: Write Silver Declaration with Expectations ~5 min

Complete the Silver layer SQL with data quality constraints.

**What you need to do:** Fill in `ON VIOLATION ________` → `DROP ROW` for both constraints.

**Guidance — Task 02**

The goal is to add **data quality expectations** to the Silver layer — the first line of defense against bad data.

**How expectations work**
Expectations are `CONSTRAINT` declarations in the `CREATE` statement. Each constraint has a name, a boolean expression, and an action. The action `ON VIOLATION DROP ROW` silently removes rows that fail the check, while `ON VIOLATION FAIL UPDATE` stops the entire pipeline. Omitting the action clause logs the violation but keeps the row.

**Choosing the right action**
Use `DROP ROW` for rules where bad data should be filtered (e.g., null IDs, negative amounts). Use `FAIL UPDATE` for critical rules where processing bad data would cause serious downstream issues (e.g., regulatory compliance). The pipeline's Data Quality tab shows violation counts.

**Things to think about**
- What happens to dropped rows — are they lost forever or can you recover them?
- Could you combine multiple conditions in a single CONSTRAINT?

In [ ]:
# TODO: Complete Silver SQL with expectations
silver_sql = """
CREATE OR REFRESH STREAMING TABLE silver_orders (
    CONSTRAINT valid_id EXPECT (order_id IS NOT NULL) ON VIOLATION ________,
    CONSTRAINT positive_amount EXPECT (total_price > 0) ON VIOLATION ________
)
AS SELECT 
    order_id,
    customer_id,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(total_price AS DOUBLE) AS total_price,
    CAST(order_date AS DATE) AS order_date,
    payment_method,
    store_id,
    current_timestamp() AS processed_at
FROM STREAM(bronze_orders);
"""

print("Silver SQL declaration:")
print(silver_sql)

In [ ]:
# -- Validation --
assert "CONSTRAINT" in silver_sql.upper(), "Should have CONSTRAINT declarations"
assert "DROP ROW" in silver_sql.upper(), "Should use ON VIOLATION DROP ROW"
assert "bronze" in silver_sql.lower(), "Should reference bronze_orders"
print("Task 2 OK: Silver declaration with expectations correct")

---
## Task 3: Write Gold Declaration ~4 min

Create a Materialized View for daily revenue summary.

**What you need to do:** Fill in `CREATE OR REFRESH ________ ________` → `MATERIALIZED VIEW`

**Guidance — Task 03**

The goal is to write a **Gold Materialized View** — the final, aggregated layer optimized for analytics.

**MATERIALIZED VIEW vs STREAMING TABLE**
A `MATERIALIZED VIEW` is recomputed from scratch on each pipeline run — Lakeflow re-reads the entire source table, applies the query, and replaces the result. This makes it perfect for aggregations (`SUM`, `COUNT`, `GROUP BY`) where the result depends on the full dataset. A `STREAMING TABLE` only processes new data, which doesn't work for aggregations over historical data.

**When to use each**
Use `STREAMING TABLE` for append-only transformations (Bronze, Silver), and `MATERIALIZED VIEW` for aggregated or joined Gold tables. Lakeflow optimizes materialized views internally — it may skip recomputation if it detects the source hasn't changed.

**Things to think about**
- Why can't a `GROUP BY` aggregation be done incrementally as a streaming table?
- When might you choose a regular VIEW over a MATERIALIZED VIEW in Lakeflow?

In [ ]:
# TODO: Complete Gold declaration
gold_sql = """
CREATE OR REFRESH ________ ________ gold_daily_revenue
AS SELECT 
    order_date,
    SUM(total_price) AS total_revenue,
    COUNT(*) AS total_orders,
    AVG(total_price) AS avg_order_value
FROM silver_orders
GROUP BY order_date
ORDER BY order_date;
"""

print("Gold SQL declaration:")
print(gold_sql)

In [ ]:
# -- Validation --
assert "MATERIALIZED VIEW" in gold_sql.upper(), "Gold should use MATERIALIZED VIEW"
assert "silver" in gold_sql.lower(), "Should reference silver_orders"
print("Task 3 OK: Gold Materialized View declaration correct")

---
## Task 4: Compare STREAMING TABLE vs MATERIALIZED VIEW ~3 min

Fill in the comparison table (markdown exercise).

| Feature | STREAMING TABLE | MATERIALIZED VIEW |
|---------|----------------|-------------------|
| Processing mode | ________ | ________ |
| Best for | ________ | ________ |
| Read from source | `STREAM(table_name)` | `table_name` |
| Supports expectations | Yes | Yes |

**Guidance — Task 04**

The goal is to solidify your understanding of the **difference between STREAMING TABLE and MATERIALIZED VIEW** — a core Lakeflow concept.

**Processing mode**
STREAMING TABLE processes **incrementally** — only new records since the last run. MATERIALIZED VIEW processes the **complete dataset** every time it refreshes. This fundamental difference drives when you use each.

**Best for…**
Streaming tables excel at high-throughput, append-heavy workloads: raw ingestion (Bronze), row-level transformations (Silver). Materialized views excel at aggregations, joins across dimensions, and any query where the result depends on the full history.

**Things to think about**
- What happens when you do a "Full Refresh" on a STREAMING TABLE — does it lose checkpoint state?
- Can a MATERIALIZED VIEW read from a STREAMING TABLE?

---
## Task 5: Verify Pipeline Results ~5 min

After creating and running the pipeline via the UI, query the results here.

**What you need to do:** Uncomment the queries and run them to verify Bronze, Silver, and Gold tables exist with data.

**Guidance — Task 05**

The goal is to **verify the pipeline ran correctly** by querying the output tables in each medallion layer.

**Where pipeline tables live**
When you configured the pipeline, you set a Target Schema. All tables declared in the pipeline are created under `catalog.schema.table_name`. The Bronze, Silver, and Gold tables are normal Delta tables — you query them with standard SQL.

**What to look for**
- Bronze should have raw records matching the number of input files
- Silver should have fewer rows if expectations dropped bad data
- Gold should have aggregated results (one row per date)

**Things to think about**
- If Bronze has 100 rows but Silver has only 95 — how do you find which 5 were dropped?
- Can you query pipeline tables from outside the pipeline (e.g., in a notebook)?

In [ ]:
# TODO: After running the pipeline, uncomment and verify:

# display(spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.bronze.bronze_orders"))
# display(spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.silver.silver_orders"))
# display(spark.sql(f"SELECT * FROM {CATALOG}.gold.gold_daily_revenue ORDER BY order_date"))

---
## Task 6: Check Pipeline Event Log ~4 min

Query the pipeline event log to see data quality metrics.

**What you need to do:** Uncomment the query and check how many rows were dropped by the Silver expectations.

**Guidance — Task 06**

The goal is to query the **pipeline Event Log** — a built-in audit trail that records every pipeline action.

**What the Event Log captures**
Every pipeline run generates events: schema inference, data quality checks, row counts, errors, and lineage. The `event_log()` table function exposes these as a queryable Delta table. The `details` column is a JSON struct with nested fields.

**Navigating the JSON structure**
Use Spark's `:` (colon) syntax for JSON path access: `details:flow_progress:data_quality:expectations`. This extracts the nested expectations array showing how many rows passed vs. failed each constraint.

**Things to think about**
- How would you set up an alert when data quality violations exceed a threshold?
- What other information is in the Event Log besides data quality?

In [ ]:
# TODO: After running the pipeline, uncomment to check event log:

# display(spark.sql("""
#     SELECT timestamp, details:flow_progress:data_quality:expectations
#     FROM event_log(TABLE({CATALOG}.silver.silver_orders))
#     WHERE details:flow_progress:data_quality IS NOT NULL
# """))

---
## Lab Complete!

You have:
- Written Bronze STREAMING TABLE declarations
- Written Silver declarations with data quality expectations
- Written Gold MATERIALIZED VIEW declarations
- Understood ST vs MV differences
- (If pipeline ran) Verified results and checked data quality metrics

> **Pro Tip:** In Spark Declarative Pipelines, tables within the same pipeline reference each other directly by name — no prefix needed. Use `STREAM(table_name)` for streaming reads and just `table_name` for batch reads.

> **Next:** Return to the [04 -- Orchestration Demo](../Demo/04_orchestration_demo.ipynb)

## Cleanup (Optional)

In [ ]:
# Pipeline cleanup is done via Lakeflow UI (delete the pipeline)
print("Workshop complete. Delete the pipeline from Lakeflow UI when done.")

---

← [03 — Lakeflow Pipelines](../Demo/03_lakeflow_pipelines_demo.ipynb) | **[ README](../../../README.md)** | [04 — Orchestration](../Demo/04_orchestration_demo.ipynb) →
